# RLlib — Scalable Reinforcement Learning on Ray

---

## What Is This Notebook About?

**RLlib** is a production-grade, distributed reinforcement learning library built on top of **Ray** — a framework for parallel and distributed Python. If Stable-Baselines3 is like a personal trainer, RLlib is like an entire sports training facility: it can train hundreds of agents in parallel across dozens of machines, handles complex multi-agent scenarios, and scales from a laptop to a cloud cluster without changing your code.

By the end of this notebook you will understand:
- Why distributed RL matters (scale and speed)
- The Ray ecosystem and how RLlib fits in
- RLlib's Algorithm API (train, evaluate, checkpoint)
- Multi-agent RL concepts
- Custom environments and policies with RLlib
- When to use RLlib vs Stable-Baselines3
- A mini-project: design a multi-agent competitive game

---

## Real-World Analogy: Factory vs Garage

Imagine training AI for a self-driving car:
- **Stable-Baselines3** = building a car in your garage. Perfect for learning and small projects.
- **RLlib** = running a car factory with 1000 workers. Each worker tests different road scenarios simultaneously. Results are aggregated centrally. Factory produces 1000× more data per day.

Netflix, Google, and OpenAI use distributed RL (like RLlib) because:
- Single-environment training takes weeks for complex tasks
- Running 100 parallel environments cuts training to hours
- Multi-agent games need multiple agents learning simultaneously

---

## The Ray Ecosystem

```
Ray Core ─────────────────────── Distributed execution primitives
  ├── Ray Tune ─────────────── Hyperparameter optimization at scale
  ├── Ray Train ────────────── Distributed model training (PyTorch/TF)
  ├── Ray Serve ────────────── Model serving and deployment
  ├── Ray Data ─────────────── Distributed data processing
  └── RLlib ────────────────── Distributed reinforcement learning
```

RLlib can use **Ray Tune** for hyperparameter search and **Ray Serve** for deploying trained policies — the whole pipeline is integrated.

---

## Prerequisites
- Gymnasium notebook (RL environments)
- Stable-Baselines3 notebook (algorithm concepts: PPO, DQN)
- Basic Python class concepts

---

## Table of Contents
1. Installation & Setup
2. Why Distributed RL?
3. RLlib Architecture
4. Training with the Algorithm API
5. RLlib vs Stable-Baselines3
6. Ray Tune — Hyperparameter Search
7. Multi-Agent RL Concepts
8. Custom Environments with RLlib
9. Common Pitfalls
10. Mini Project: Multi-Agent Design
11. Interview Q&A
12. Resources

---

## Official Resources
- **RLlib Docs**: https://docs.ray.io/en/latest/rllib/index.html
- **Ray GitHub**: https://github.com/ray-project/ray
- **RLlib Paper**: https://arxiv.org/abs/1712.09381
- **Ray YouTube**: https://www.youtube.com/@RayDistributed
- **RLlib Algorithm Docs**: https://docs.ray.io/en/latest/rllib/rllib-algorithms.html

## 1. Installation & Setup

In [ ]:
# Install commands:
# pip install 'ray[rllib,tune]'      # RLlib + Ray Tune
# pip install gymnasium torch

try:
    import ray
    from ray import tune
    from ray.rllib.algorithms.ppo import PPOConfig
    from ray.rllib.algorithms.dqn import DQNConfig
    RLLIB_AVAILABLE = True
    print(f"Ray version: {ray.__version__}")
    print(f"RLlib available!")
except ImportError:
    RLLIB_AVAILABLE = False
    print("RLlib not installed. Run: pip install 'ray[rllib,tune]'")
    print("All cells simulate output for learning purposes.")

try:
    import gymnasium as gym
    GYM_AVAILABLE = True
except ImportError:
    GYM_AVAILABLE = False

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

print(f"\nRLlib: {'✓' if RLLIB_AVAILABLE else '✗'}  Gymnasium: {'✓' if GYM_AVAILABLE else '✗'}")
print("Setup complete!")

## 2. Why Distributed RL?

### The Scale Problem

Training a simple CartPole agent with PPO takes ~50k timesteps. Training AlphaStar (StarCraft II) took:
- **200 years of game experience** (compressed into weeks of real time)
- **16 Google TPUs + 3000 CPU cores** running in parallel
- Millions of self-play games simultaneously

You cannot do this on a single machine with a single environment. You need:
1. **Many parallel environments** — each on its own CPU core
2. **Many workers** — each running a group of environments
3. **Centralized learning** — aggregate gradients/experiences from all workers
4. **Distributed storage** — replay buffers and model parameters

RLlib provides all of this with a unified API.

In [ ]:
# ── Distributed RL: Speed vs Correctness Trade-off Visualization ──────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Time to solve CartPole as function of workers
ax = axes[0]
n_workers = [1, 2, 4, 8, 16, 32]
# Approximate: time decreases sublinearly due to coordination overhead
time_to_solve = [100, 58, 32, 18, 11, 7.5]  # minutes (simulated)
ax.plot(n_workers, time_to_solve, 'bo-', linewidth=2.5, markersize=8)
ax.fill_between(n_workers, time_to_solve, alpha=0.2, color='blue')

# Ideal linear scaling
ideal = [100 / n for n in n_workers]
ax.plot(n_workers, ideal, 'r--', linewidth=1.5, label='Ideal linear scaling', alpha=0.7)

ax.set_title('Training Time vs Number of Workers\n(CartPole-v1, 50k steps)', fontweight='bold')
ax.set_xlabel('Number of Parallel Workers')
ax.set_ylabel('Time to Solve (minutes)')
ax.legend()
ax.grid(True, alpha=0.3)

for n, t in zip(n_workers, time_to_solve):
    ax.annotate(f'{t:.1f}m', (n, t), textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=9)

# Right: Architecture diagram
ax2 = axes[1]
ax2.axis('off')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)

# Learner (central)
learner = mpatches.FancyBboxPatch((3.5, 7.5), 3, 1.5, boxstyle='round,pad=0.1',
                                    facecolor='gold', edgecolor='black', linewidth=2)
ax2.add_patch(learner)
ax2.text(5, 8.25, 'Learner\n(Central GPU)', ha='center', va='center', fontsize=11, fontweight='bold')

# Workers
worker_positions = [(0.5, 4), (2.5, 4), (4.5, 4), (6.5, 4), (8.5, 4)]
colors = plt.cm.tab10(np.linspace(0, 1, 5))
for i, (x, y) in enumerate(worker_positions):
    w = mpatches.FancyBboxPatch((x, y), 1.5, 1.2, boxstyle='round,pad=0.05',
                                  facecolor=colors[i], edgecolor='black', linewidth=1.5, alpha=0.8)
    ax2.add_patch(w)
    ax2.text(x+0.75, y+0.6, f'Worker {i+1}\n4 envs', ha='center', va='center',
             fontsize=9, fontweight='bold')
    # Arrow: worker → learner (gradients/samples)
    ax2.annotate('', xy=(5, 7.5), xytext=(x+0.75, y+1.2),
                 arrowprops=dict(arrowstyle='->', color=colors[i], lw=1.5))
    # Arrow: learner → worker (updated weights)
    ax2.annotate('', xy=(x+0.75, y+1.2), xytext=(5, 7.5),
                 arrowprops=dict(arrowstyle='->', color='gray', lw=1, linestyle='dashed'))

# Env boxes under each worker
for i, (x, y) in enumerate(worker_positions):
    for j in range(4):
        env = mpatches.FancyBboxPatch((x + j*0.35, y-1.8), 0.3, 0.8,
                                       boxstyle='round,pad=0.02',
                                       facecolor='lightblue', edgecolor='gray', linewidth=1)
        ax2.add_patch(env)
        ax2.text(x + j*0.35 + 0.15, y-1.4, f'E', ha='center', va='center', fontsize=6)
    ax2.annotate('', xy=(x+0.75, y), xytext=(x+0.75, y-1.0),
                 arrowprops=dict(arrowstyle='->', color='gray', lw=1))

ax2.text(5, 9.5, 'RLlib Distributed Architecture:\n5 Workers × 4 Envs = 20 Parallel Environments',
         ha='center', va='center', fontsize=11, fontweight='bold')
ax2.text(7.5, 8.0, '◄ Weights (broadcast)', fontsize=8, color='gray', style='italic')
ax2.text(7.5, 7.2, '► Samples/Gradients', fontsize=8, color='navy', style='italic')

plt.suptitle('Why Distributed RL: Parallelism = Speed', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/rllib_architecture.png', dpi=100, bbox_inches='tight')
plt.show()

print("With 5 workers × 4 envs = 20 parallel environments:")
print("  Data collection: 20× faster")
print("  But: coordination overhead reduces practical speedup to ~8-15×")
print("  Still: for complex tasks (robotics, games), this is essential")

## 3. RLlib Architecture

RLlib organizes the RL training loop into distinct components:

```
Algorithm (e.g., PPO)
│
├── RolloutWorkers (data collection)
│     ├── Worker 1: [Env1, Env2, Env3, Env4] → collect trajectories
│     ├── Worker 2: [Env5, Env6, Env7, Env8] → collect trajectories
│     └── Worker N: [...]
│
├── ReplayBuffer (off-policy only: DQN, SAC)
│     └── Stores transitions; workers write, learner reads
│
├── Learner (model update)
│     └── Receives batches → computes gradients → updates weights
│
└── Policy
      └── Neural network: obs → action probabilities (or Q-values)
```

### RLlib Config System
RLlib uses a **Config builder pattern**. You build a config object, then call `.build()` to create the algorithm:

```python
config = (
    PPOConfig()
    .environment('CartPole-v1')
    .rollouts(num_rollout_workers=4)
    .training(lr=3e-4, gamma=0.99)
    .resources(num_gpus=1)
)
algo = config.build()
```

Each method returns `self`, enabling method chaining.

In [ ]:
# ── RLlib Training with Algorithm API ─────────────────────────────────

if RLLIB_AVAILABLE:
    # Initialize Ray (uses all available CPUs)
    if not ray.is_initialized():
        ray.init(num_cpus=4, ignore_reinit_error=True)

    # Build PPO config
    config = (
        PPOConfig()
        .environment('CartPole-v1')
        .rollouts(
            num_rollout_workers=2,    # 2 parallel data collection workers
            num_envs_per_worker=2,    # 2 envs per worker = 4 total
        )
        .training(
            lr=3e-4,
            gamma=0.99,
            train_batch_size=512,     # Collect 512 steps before updating
            sgd_minibatch_size=128,
            num_sgd_iter=10,          # PPO epochs
        )
        .framework('torch')           # Use PyTorch backend
        .resources(num_gpus=0)        # CPU only
    )

    # Build the algorithm
    algo = config.build()

    print("Training PPO with RLlib...")
    print(f"Workers: 2 × 2 envs = 4 parallel environments")
    print()

    rewards = []
    for iteration in range(5):
        result = algo.train()  # One training iteration
        mean_r = result['episode_reward_mean']
        rewards.append(mean_r)
        print(f"Iter {iteration+1:2d}: "
              f"mean_reward={mean_r:7.1f}, "
              f"episodes={result['episodes_total']:4d}, "
              f"timesteps={result['timesteps_total']:6d}")

    # Save checkpoint
    checkpoint = algo.save('/tmp/rllib_checkpoint')
    print(f"\nCheckpoint saved: {checkpoint}")

    algo.stop()
    ray.shutdown()

else:
    print("=== RLlib PPO Training (simulated) ===")
    print()
    print("import ray")
    print("from ray.rllib.algorithms.ppo import PPOConfig")
    print()
    print("ray.init(num_cpus=4)")
    print()
    print("config = (")
    print("    PPOConfig()")
    print("    .environment('CartPole-v1')")
    print("    .rollouts(num_rollout_workers=2, num_envs_per_worker=2)")
    print("    .training(lr=3e-4, gamma=0.99, train_batch_size=512)")
    print("    .framework('torch')")
    print(")")
    print()
    print("algo = config.build()")
    print()
    print("Output:")
    simulated_rewards = [24.3, 67.1, 134.5, 223.8, 318.4]
    for i, (r, eps, ts) in enumerate(zip(simulated_rewards, [41,96,178,271,365], [512,1024,1536,2048,2560])):
        print(f"Iter {i+1:2d}: mean_reward={r:7.1f}, episodes={eps:4d}, timesteps={ts:6d}")
    print()
    print("checkpoint_dir = algo.save('/tmp/rllib_checkpoint')")
    print("Checkpoint saved: /tmp/rllib_checkpoint/checkpoint_000005")
    rewards = simulated_rewards

## 4. RLlib vs Stable-Baselines3 — When to Use Which

In [ ]:
# ── Comparison Table ──────────────────────────────────────────────────

print("=" * 75)
print(" RLlib vs Stable-Baselines3: Feature Comparison")
print("=" * 75)

comparisons = [
    ('Feature', 'Stable-Baselines3', 'RLlib'),
    ('─' * 30, '─' * 22, '─' * 22),
    ('Ease of setup', '⭐⭐⭐⭐⭐ Very easy', '⭐⭐⭐ Moderate'),
    ('Scalability', '⭐⭐ Single machine', '⭐⭐⭐⭐⭐ Cluster scale'),
    ('Multi-agent RL', '✗ Not supported', '✓ First-class support'),
    ('Custom policies', '⭐⭐⭐ Moderate', '⭐⭐⭐⭐ Extensive'),
    ('Algorithm variety', '⭐⭐⭐ PPO/DQN/SAC/TD3', '⭐⭐⭐⭐⭐ 30+ algorithms'),
    ('Documentation', '⭐⭐⭐⭐ Excellent', '⭐⭐⭐ Good (complex)'),
    ('Debugging', '⭐⭐⭐⭐ Easy', '⭐⭐ Hard (distributed)'),
    ('Hyperparameter tuning', '⭐⭐ Manual/Optuna', '⭐⭐⭐⭐⭐ Ray Tune built-in'),
    ('Production deployment', '⭐⭐⭐ Manual', '⭐⭐⭐⭐⭐ Ray Serve integration'),
    ('Best for', 'Research, learning, prototypes', 'Production, large scale, multi-agent'),
]

for row in comparisons:
    print(f"{row[0]:<32} {row[1]:<28} {row[2]:<28}")

print("\n" + "=" * 75)
print("\nDecision Guide:")
print("  Use SB3 if:")
print("    - Learning RL or prototyping")
print("    - Single-agent, single-machine training")
print("    - Want clean, simple code")
print("    - Task solves in <1M timesteps")
print()
print("  Use RLlib if:")
print("    - Training at scale (millions of timesteps, multiple machines)")
print("    - Multi-agent scenarios (competitive or cooperative)")
print("    - Need Ray Tune for hyperparameter search at scale")
print("    - Building production RL systems")
print("    - Need hierarchical or goal-conditioned RL")

## 5. Ray Tune — Hyperparameter Search at Scale

Finding the best learning rate, gamma, clip range, etc. manually is tedious. **Ray Tune** automates this by running many trials in parallel and using smart search algorithms (Bayesian optimization, Hyperband) to find the best configuration.

### Search Algorithms Available:
| Algorithm | Description |
|-----------|-------------|
| Grid Search | Try all combinations (exhaustive but slow) |
| Random Search | Random combinations (surprisingly effective) |
| Bayesian Optimization | Smart search based on past results |
| Hyperband (ASHA) | Kill bad trials early, focus on promising ones |
| Population Based Training | Evolve hyperparameters during training |

**ASHA (Asynchronous Successive Halving)** is the most common: start 100 trials, after 1000 steps kill the bottom 50%, keep the top 50%, continue, repeat. Very efficient.

In [ ]:
# ── Ray Tune Hyperparameter Search ────────────────────────────────────

if RLLIB_AVAILABLE:
    from ray import tune
    from ray.tune.schedulers import ASHAScheduler

    if not ray.is_initialized():
        ray.init(num_cpus=4, ignore_reinit_error=True)

    # Define search space
    param_space = (
        PPOConfig()
        .environment('CartPole-v1')
        .training(
            # tune.loguniform = sample uniformly in log-scale
            lr=tune.loguniform(1e-5, 1e-3),
            gamma=tune.choice([0.9, 0.95, 0.99, 0.999]),
            train_batch_size=tune.choice([256, 512, 1024]),
        )
        .rollouts(num_rollout_workers=1)
        .framework('torch')
        .to_dict()
    )

    # Run 5 trials (would normally run 50-200)
    tuner = tune.Tuner(
        'PPO',
        param_space=param_space,
        tune_config=tune.TuneConfig(
            metric='episode_reward_mean',
            mode='max',
            num_samples=5,             # Number of hyperparameter trials
            scheduler=ASHAScheduler(   # Kill bad trials early
                max_t=5,               # Max iterations per trial
                grace_period=1,
            )
        ),
        run_config=tune.RunConfig(
            stop={'training_iteration': 3}
        )
    )

    results = tuner.fit()
    best = results.get_best_result()
    print(f"\nBest config:")
    print(f"  lr:               {best.config['lr']:.2e}")
    print(f"  gamma:            {best.config['gamma']}")
    print(f"  train_batch_size: {best.config['train_batch_size']}")
    print(f"  Best reward:      {best.metrics['episode_reward_mean']:.1f}")

    ray.shutdown()

else:
    print("=== Ray Tune Hyperparameter Search (code + simulated output) ===")
    print()
    code = '''
from ray import tune
from ray.tune.schedulers import ASHAScheduler

# Define the search space — Tune samples from these distributions
param_space = (
    PPOConfig()
    .environment("CartPole-v1")
    .training(
        lr=tune.loguniform(1e-5, 1e-3),       # Try LRs from 0.00001 to 0.001
        gamma=tune.choice([0.9, 0.95, 0.99]),  # Try 3 discount factors
        train_batch_size=tune.choice([256, 512, 1024])
    )
    .to_dict()
)

tuner = tune.Tuner(
    "PPO",
    param_space=param_space,
    tune_config=tune.TuneConfig(
        metric="episode_reward_mean",
        mode="max",
        num_samples=50,            # Run 50 trials in parallel
        scheduler=ASHAScheduler(   # Kill bad trials early
            max_t=100,             # Max training iterations
            grace_period=5,        # Min iterations before killing
        )
    )
)

results = tuner.fit()
best = results.get_best_result()
    '''
    print(code)
    print("Output (after ~30 minutes with 50 trials):")
    print("  Trial 003: lr=0.00031, gamma=0.99, batch=1024 → reward=287.3  ← BEST")
    print("  Trial 017: lr=0.00008, gamma=0.95, batch=512  → reward=214.6")
    print("  Trial 031: lr=0.00082, gamma=0.90, batch=256  → reward=87.4   (killed early)")
    print("  Trial 049: lr=0.00021, gamma=0.99, batch=512  → reward=251.1")
    print()
    print("Best config: lr=3.1e-4, gamma=0.99, train_batch_size=1024")
    print("Best reward: 287.3")

# Visualize ASHA scheduler behavior
fig, ax = plt.subplots(figsize=(10, 5))
np.random.seed(42)

# Simulate 20 trials with different final performances
n_trials = 20
n_steps = 20
final_rewards = np.random.uniform(20, 300, n_trials)

colors = plt.cm.RdYlGn(final_rewards / 300)

for trial in range(n_trials):
    # Each trial has a learning curve
    trajectory = np.linspace(20, final_rewards[trial], n_steps)
    noise = np.random.normal(0, 20, n_steps)
    trajectory = trajectory + noise

    # ASHA kills bottom 50% at step 5
    kill_step = None
    if final_rewards[trial] < np.percentile(final_rewards, 50):
        kill_step = np.random.randint(4, 8)
        ax.plot(range(kill_step), trajectory[:kill_step], color=colors[trial], alpha=0.4, linewidth=1)
        ax.axvline(kill_step, color=colors[trial], alpha=0.2, linestyle=':')
        ax.plot(kill_step-1, trajectory[kill_step-1], 'rx', markersize=8)
    else:
        ax.plot(range(n_steps), trajectory, color=colors[trial], alpha=0.8, linewidth=1.5)

# Highlight best trial
best_idx = np.argmax(final_rewards)
best_traj = np.linspace(20, final_rewards[best_idx], n_steps) + np.random.normal(0, 15, n_steps)
ax.plot(range(n_steps), best_traj, 'gold', linewidth=3, label=f'Best trial (reward={final_rewards[best_idx]:.0f})')

ax.axvline(5, color='black', linestyle='--', linewidth=1.5, label='ASHA pruning point')
ax.set_title('Ray Tune ASHA Scheduler: Kill Bad Trials Early', fontweight='bold')
ax.set_xlabel('Training Iteration')
ax.set_ylabel('Mean Reward')
ax.legend()
ax.grid(True, alpha=0.3)
ax.text(5.2, 280, 'Bottom 50% killed here\n→ Resources freed for\n   promising trials', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('/tmp/rllib_tune_asha.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Multi-Agent RL — Multiple Agents in One Environment

Multi-Agent RL (MARL) is where multiple agents interact in the same environment, either:
- **Cooperative**: All agents share a reward, work together (like a soccer team)
- **Competitive**: Agents have opposing rewards (chess, poker)
- **Mixed**: Some cooperation, some competition (real-world traffic)

MARL is much harder than single-agent RL because:
1. The environment is **non-stationary** from each agent's perspective (other agents change)
2. Credit assignment: which agent contributed to the team reward?
3. Coordination emerges but must be incentivized

### RLlib MARL API
RLlib natively supports multi-agent environments through its `MultiAgentEnv` interface and policy mapping:

```python
# Map agents to policies
config.multi_agent(
    policies={
        'predator_policy': PolicySpec(policy_class=PPOTorchPolicy),
        'prey_policy': PolicySpec(policy_class=PPOTorchPolicy),
    },
    policy_mapping_fn=lambda agent_id, **kwargs:
        'predator_policy' if 'predator' in agent_id else 'prey_policy'
)
```

In [ ]:
# ── Multi-Agent Environment Design ────────────────────────────────────
#
# We'll design (not train) a simple multi-agent grid environment
# to illustrate the RLlib MultiAgentEnv interface.

if GYM_AVAILABLE:
    from gymnasium import spaces as gym_spaces

    class SimplePredatorPreyEnv:
        """
        A 10x10 grid where:
        - 2 Predators (cooperative) try to CATCH the prey
        - 1 Prey tries to ESCAPE the predators

        This is a classic MARL benchmark.

        RLlib MultiAgentEnv contract:
        - reset() → dict of {agent_id: observation}
        - step(actions) → obs_dict, rew_dict, terminated_dict, truncated_dict, info_dict
        - Each agent acts simultaneously
        """

        def __init__(self):
            self.grid_size = 10
            self.agents = ['predator_0', 'predator_1', 'prey_0']
            self.max_steps = 100

            # Each agent observes [own_x, own_y, other_x, other_y, other_x, other_y]
            self.observation_space = {
                agent: gym_spaces.Box(0, self.grid_size-1, shape=(6,), dtype=np.float32)
                for agent in self.agents
            }
            # 4 directions: up, down, left, right
            self.action_space = {
                agent: gym_spaces.Discrete(4)
                for agent in self.agents
            }
            self._action_to_dir = {0: (-1,0), 1: (1,0), 2: (0,-1), 3: (0,1)}

        def reset(self, seed=None):
            if seed is not None:
                np.random.seed(seed)
            # Random starting positions
            self.positions = {
                'predator_0': list(np.random.randint(0, 5, 2)),
                'predator_1': list(np.random.randint(0, 5, 2)),
                'prey_0':     list(np.random.randint(5, 10, 2)),
            }
            self.step_count = 0
            return self._get_obs(), {a: {} for a in self.agents}

        def _get_obs(self):
            """Each agent observes ALL positions (full observability)."""
            obs = {}
            for agent in self.agents:
                own = self.positions[agent]
                others = [self.positions[a] for a in self.agents if a != agent]
                obs[agent] = np.array(own + others[0] + others[1], dtype=np.float32)
            return obs

        def step(self, actions):
            """All agents act simultaneously."""
            self.step_count += 1

            # Move all agents
            for agent, action in actions.items():
                dr, dc = self._action_to_dir[action]
                r, c = self.positions[agent]
                self.positions[agent] = [
                    np.clip(r + dr, 0, self.grid_size-1),
                    np.clip(c + dc, 0, self.grid_size-1)
                ]

            # Check if prey is caught (predator within distance 1)
            prey_pos = np.array(self.positions['prey_0'])
            caught = False
            for pred_id in ['predator_0', 'predator_1']:
                pred_pos = np.array(self.positions[pred_id])
                if np.abs(prey_pos - pred_pos).sum() <= 1:  # Manhattan distance
                    caught = True
                    break

            # Rewards
            if caught:
                rewards = {'predator_0': +10, 'predator_1': +10, 'prey_0': -10}
                terminated = {a: True for a in self.agents}
            else:
                # Small step penalty for predators, bonus for prey surviving
                rewards = {'predator_0': -0.1, 'predator_1': -0.1, 'prey_0': +0.1}
                terminated = {a: False for a in self.agents}

            truncated = {a: self.step_count >= self.max_steps for a in self.agents}
            terminated['__all__'] = caught
            truncated['__all__'] = self.step_count >= self.max_steps

            return self._get_obs(), rewards, terminated, truncated, {a: {} for a in self.agents}


    # Test the multi-agent environment
    env = SimplePredatorPreyEnv()
    obs, info = env.reset(seed=42)

    print("=== Multi-Agent Predator-Prey Environment ===")
    print(f"Agents: {env.agents}")
    print(f"\nInitial positions:")
    for agent, pos in env.positions.items():
        print(f"  {agent}: {pos}")
    print(f"\nObservation shapes:")
    for agent, ob in obs.items():
        print(f"  {agent}: {ob}  (own_pos + 2 other positions)")

    # Run 5 steps with random actions
    print("\n5 random steps:")
    total_pred_reward = 0
    total_prey_reward = 0

    for step in range(5):
        actions = {agent: env.action_space[agent].sample() for agent in env.agents}
        obs, rewards, terminated, truncated, info = env.step(actions)
        total_pred_reward += rewards['predator_0']
        total_prey_reward += rewards['prey_0']
        caught = terminated.get('__all__', False)
        print(f"  Step {step+1}: pred0_pos={env.positions['predator_0']}, "
              f"prey_pos={env.positions['prey_0']}, "
              f"caught={caught}")

    print(f"\nCumulative: pred reward={total_pred_reward:.1f}, prey reward={total_prey_reward:.1f}")

else:
    print("=== Multi-Agent Environment (simulated output) ===")
    print("Agents: ['predator_0', 'predator_1', 'prey_0']")
    print()
    print("Initial positions:")
    print("  predator_0: [2, 1]")
    print("  predator_1: [3, 4]")
    print("  prey_0:     [7, 8]")
    print()
    print("5 random steps:")
    for i in range(5):
        print(f"  Step {i+1}: pred0_pos=[{2+i},{1+i}], prey_pos=[{7},{8-i}], caught=False")
    print("Cumulative: pred reward=-0.5, prey reward=+0.5")

In [ ]:
# ── Visualize a Predator-Prey Episode ─────────────────────────────────

def visualize_predator_prey_episode(n_steps=30):
    """Simulate and visualize a predator-prey episode."""
    GRID = 10

    # Simulate simple heuristic behavior for visualization
    np.random.seed(42)

    pred0 = np.array([1.0, 1.0])
    pred1 = np.array([2.0, 0.0])
    prey = np.array([8.0, 8.0])

    pred0_history = [pred0.copy()]
    pred1_history = [pred1.copy()]
    prey_history = [prey.copy()]

    for _ in range(n_steps):
        # Predators move toward prey (simple heuristic)
        def move_toward(agent, target, speed=0.6):
            direction = target - agent
            if np.linalg.norm(direction) > 0:
                direction = direction / np.linalg.norm(direction)
            noise = np.random.normal(0, 0.2, 2)
            new_pos = agent + direction * speed + noise
            return np.clip(new_pos, 0, GRID-1)

        def flee(agent, threats, speed=0.7):
            avg_threat = np.mean(threats, axis=0)
            direction = agent - avg_threat
            if np.linalg.norm(direction) > 0:
                direction = direction / np.linalg.norm(direction)
            noise = np.random.normal(0, 0.3, 2)
            new_pos = agent + direction * speed + noise
            return np.clip(new_pos, 0, GRID-1)

        pred0 = move_toward(pred0, prey)
        pred1 = move_toward(pred1, prey)
        prey = flee(prey, [pred0, pred1])

        pred0_history.append(pred0.copy())
        pred1_history.append(pred1.copy())
        prey_history.append(prey.copy())

        if np.linalg.norm(prey - pred0) < 1 or np.linalg.norm(prey - pred1) < 1:
            break

    # Plot trajectories
    fig, ax = plt.subplots(figsize=(8, 8))

    pred0_arr = np.array(pred0_history)
    pred1_arr = np.array(pred1_history)
    prey_arr = np.array(prey_history)

    ax.plot(pred0_arr[:, 0], pred0_arr[:, 1], 'r-', linewidth=2, alpha=0.7, label='Predator 0')
    ax.plot(pred1_arr[:, 0], pred1_arr[:, 1], 'b-', linewidth=2, alpha=0.7, label='Predator 1')
    ax.plot(prey_arr[:, 0], prey_arr[:, 1], 'g-', linewidth=2, alpha=0.7, label='Prey')

    # Start/end markers
    ax.plot(*pred0_arr[0], 'rs', markersize=12, label='Pred 0 start')
    ax.plot(*pred1_arr[0], 'bs', markersize=12, label='Pred 1 start')
    ax.plot(*prey_arr[0], 'g^', markersize=12, label='Prey start')
    ax.plot(*pred0_arr[-1], 'ro', markersize=10)
    ax.plot(*pred1_arr[-1], 'bo', markersize=10)
    ax.plot(*prey_arr[-1], 'g*', markersize=15)

    # Step numbers
    for i in range(0, len(prey_arr), 5):
        ax.text(prey_arr[i, 0]+0.2, prey_arr[i, 1]+0.2, str(i), fontsize=8, color='green')

    ax.set_xlim(-0.5, GRID)
    ax.set_ylim(-0.5, GRID)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(f'Multi-Agent Predator-Prey Episode\n({len(pred0_history)-1} steps until catch)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')
    ax.legend(loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.savefig('/tmp/rllib_predator_prey.png', dpi=100, bbox_inches='tight')
    plt.show()

visualize_predator_prey_episode(n_steps=30)

print("Multi-Agent Learning Challenges:")
print("  1. Non-stationarity: As predators learn, the prey has to adapt (and vice versa)")
print("  2. Coordination: Predators need to surround the prey, not both approach from the same side")
print("  3. Credit assignment: If predators cooperate and catch prey, who gets credit?")
print()
print("RLlib handles all of this through its multi-agent API!")

## 7. Common Pitfalls

In [ ]:
# ── Common RLlib Pitfalls ─────────────────────────────────────────────

print("=" * 68)
print(" Common RLlib Pitfalls & Fixes")
print("=" * 68)

pitfalls = [
    {
        "title": "1. Not calling ray.init() before using RLlib",
        "symptom": "RuntimeError: Ray has not been initialized",
        "fix": "ray.init() at the start; ray.shutdown() at the end",
        "detail": "Ray needs to start worker processes. Always init Ray first."
    },
    {
        "title": "2. Forgetting ignore_reinit_error=True in notebooks",
        "symptom": "ValueError: Ray already initialized. Shutdown and reinit.",
        "fix": "ray.init(ignore_reinit_error=True)  # Safe to call multiple times",
        "detail": "In notebooks, re-running a cell calls ray.init() again. Use ignore_reinit_error."
    },
    {
        "title": "3. Not calling algo.stop() after training",
        "symptom": "Worker processes linger, consuming memory and CPU",
        "fix": "algo.stop()  # Always stop the algorithm after use",
        "detail": "RLlib spawns multiple Ray actors. Stopping the algo shuts them down."
    },
    {
        "title": "4. RLlib API changes between versions",
        "symptom": "AttributeError or DeprecationWarning on config fields",
        "fix": "Pin Ray version: pip install 'ray[rllib]==2.x.x'",
        "detail": "RLlib's API changed significantly in v2.0. Use AlgorithmConfig builder style."
    },
    {
        "title": "5. Not registering custom envs with Ray's registry",
        "symptom": "Workers can't find the custom environment class",
        "fix": "from ray.tune.registry import register_env\n"
               "       register_env('MyEnv-v0', lambda config: MyEnvClass(config))",
        "detail": "Workers are separate processes. They need to know how to create your env."
    },
    {
        "title": "6. Allocating too many workers on a small machine",
        "symptom": "OOM errors, machine freezes, very slow training",
        "fix": "Use num_rollout_workers ≤ num_CPUs - 1 (leave 1 for the driver)",
        "detail": "Each worker is a full Python process. 8 workers on a 4-core machine = oversubscribed."
    },
]

for p in pitfalls:
    print(f"\n{'─'*68}")
    print(f"  {p['title']}")
    print(f"  Symptom: {p['symptom']}")
    print(f"  Fix:     {p['fix']}")
    print(f"  Detail:  {p['detail']}")

print(f"\n{'='*68}")

## 8. Mini Project: RLlib Algorithm Comparison Dashboard

In [ ]:
# ── Mini Project: Simulate and Compare RLlib Algorithm Results ────────
#
# In a real experiment, you'd run multiple RLlib algorithms on the same
# environment and compare their learning curves. We simulate this here
# with realistic learning dynamics.

def simulate_rllib_training(algorithm, env_name, n_iters=50, seed=42):
    """
    Simulate realistic training curves for different RLlib algorithms.
    Based on published benchmarks from the RLlib paper and SB3 Zoo.
    """
    np.random.seed(seed)

    curves = {
        'CartPole-v1': {
            'PPO':  {'final': 470, 'speed': 0.15, 'noise': 40},
            'DQN':  {'final': 420, 'speed': 0.10, 'noise': 55},
            'A2C':  {'final': 380, 'speed': 0.12, 'noise': 70},
            'IMPALA': {'final': 490, 'speed': 0.18, 'noise': 30},
        },
        'LunarLander-v2': {
            'PPO':  {'final': 240, 'speed': 0.08, 'noise': 60},
            'DQN':  {'final': 200, 'speed': 0.06, 'noise': 80},
            'A2C':  {'final': 170, 'speed': 0.07, 'noise': 90},
            'IMPALA': {'final': 260, 'speed': 0.10, 'noise': 50},
        }
    }

    params = curves[env_name][algorithm]
    rewards = []

    for i in range(n_iters):
        progress = i / n_iters
        # Sigmoid-like learning curve
        base = -200 + (params['final'] + 200) / (1 + np.exp(-params['speed'] * (i - n_iters/3)))
        noise = params['noise'] * np.exp(-2 * progress)  # Noise decreases as training matures
        rewards.append(base + np.random.normal(0, noise))

    return rewards


# Compare algorithms on CartPole
env_name = 'CartPole-v1'
algorithms = ['PPO', 'DQN', 'A2C', 'IMPALA']
colors_map = {'PPO': 'blue', 'DQN': 'red', 'A2C': 'green', 'IMPALA': 'purple'}

training_data = {}
for algo in algorithms:
    training_data[algo] = simulate_rllib_training(algo, env_name, n_iters=60, seed=algo.__hash__() % 100)

window = 10

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'RLlib Algorithm Comparison on {env_name}', fontsize=14, fontweight='bold')

# 1. Learning curves
ax = axes[0, 0]
for algo, rewards in training_data.items():
    ax.plot(rewards, alpha=0.2, color=colors_map[algo])
    smooth = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(rewards)), smooth, color=colors_map[algo],
            linewidth=2.5, label=f'{algo} (final={smooth[-1]:.0f})')
ax.axhline(475, color='gold', linestyle='--', linewidth=2, label='Solved (475)')
ax.set_title('Learning Curves', fontweight='bold')
ax.set_xlabel('Training Iteration')
ax.set_ylabel('Mean Episode Reward')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. Final performance comparison
ax = axes[0, 1]
final_scores = [np.mean(training_data[algo][-10:]) for algo in algorithms]
bars = ax.bar(algorithms, final_scores, color=[colors_map[a] for a in algorithms], alpha=0.8)
ax.axhline(475, color='gold', linestyle='--', linewidth=2, label='Solved (475)')
for bar, score in zip(bars, final_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{score:.0f}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Final Performance (Last 10 Iterations)', fontweight='bold')
ax.set_ylabel('Mean Reward')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 3. Sample efficiency (iterations to reach reward=200)
ax = axes[1, 0]
threshold = 200
iters_to_threshold = []
for algo in algorithms:
    smooth = np.convolve(training_data[algo], np.ones(window)/window, mode='valid')
    crossed = next((i for i, r in enumerate(smooth) if r >= threshold), len(smooth))
    iters_to_threshold.append(crossed)

bars = ax.barh(algorithms, iters_to_threshold, color=[colors_map[a] for a in algorithms], alpha=0.8)
for bar, iters in zip(bars, iters_to_threshold):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{iters} iters', va='center', fontsize=10)
ax.set_title(f'Iterations to Reach Reward={threshold}\n(Lower = More Sample Efficient)', fontweight='bold')
ax.set_xlabel('Iterations')
ax.grid(True, alpha=0.3, axis='x')

# 4. RLlib config comparison
ax = axes[1, 1]
ax.axis('off')
table_data = [
    ['Algorithm', 'Policy Type', 'On/Off Policy', 'Best Env Type', 'Workers Scaling'],
    ['PPO', 'Actor-Critic', 'On-Policy', 'General', 'Excellent'],
    ['DQN', 'Value-Based', 'Off-Policy', 'Discrete', 'Good'],
    ['A2C', 'Actor-Critic', 'On-Policy', 'General', 'Good'],
    ['IMPALA', 'Actor-Critic', 'Off-Policy', 'Large Scale', 'Excellent'],
    ['SAC', 'Actor-Critic', 'Off-Policy', 'Continuous', 'Moderate'],
]

table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                  cellLoc='center', loc='center',
                  colWidths=[0.15, 0.18, 0.18, 0.22, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.8)
for j in range(5):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')
ax.set_title('RLlib Algorithm Quick Reference', fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('/tmp/rllib_comparison_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nKey takeaways:")
print("  IMPALA: best for large scale (many workers, huge envs like Atari)")
print("  PPO:    best balance of performance, stability, and ease")
print("  DQN:    good sample efficiency but only for discrete actions")
print("  A2C:    faster iteration but more hyperparameter sensitivity")

## 9. Interview Q&A

---

### Q1: What is Ray and why is RLlib built on it?
**A**: Ray is a general-purpose distributed computing framework for Python. It provides: (1) **Remote functions** (`@ray.remote`): run functions on any machine in the cluster, (2) **Actors** (`@ray.remote` class): stateful workers that can run code remotely, (3) **Object store**: efficient sharing of large objects (like model weights) between workers. RLlib uses Ray to distribute data collection (rollout workers as Ray actors) and hyperparameter search (Ray Tune). The key benefit: you write single-machine Python code, and Ray runs it on a cluster with no code changes.

---

### Q2: What is IMPALA and why is it good for large-scale training?
**A**: IMPALA (Importance Weighted Actor-Learner Architecture) separates the actors (data collection) from the learner (gradient computation). Actors collect trajectories and send them to the learner asynchronously — they don't wait for the learner to finish. This allows the learner to process data continuously while actors collect new data simultaneously. It's off-policy (uses V-trace importance sampling to correct for the stale behavior policy), so it can use experiences from any time. This makes IMPALA very efficient at extreme scales (thousands of actors).

---

### Q3: How does RLlib handle multi-agent environments?
**A**: RLlib's `MultiAgentEnv` interface requires `step()` to accept a dict of `{agent_id: action}` and return dicts of `{agent_id: obs/reward/done}`. Each agent can have its own policy (policy mapping function determines which policy handles each agent). Policies can be shared (all agents use the same policy — parameter sharing) or separate. This handles cooperative, competitive, and mixed scenarios. The training loss is computed per-policy, aggregating experiences from all agents using that policy.

---

### Q4: What is Ray Tune's ASHA scheduler and how does it save time?
**A**: ASHA (Asynchronous Successive Halving Algorithm) is an early stopping strategy for hyperparameter search. It starts many trials simultaneously. At predefined checkpoints (called rungs), it keeps only the top fraction (e.g., top 50%) of trials based on their current performance, and kills the rest. This repeats until only the best trial(s) remain. The savings come from not wasting compute on clearly bad hyperparameters. Instead of each of 100 trials running for 100 iterations (10,000 total), ASHA might use ~2000 total iterations to find the same answer.

---

### Q5: RLlib vs Stable-Baselines3 — which should I use in production?
**A**: For production RL systems at scale: RLlib. Reasons: (1) Built-in distributed training — scale from 1 to 1000s of workers without code changes, (2) Ray Tune integration for automated hyperparameter optimization, (3) Ray Serve integration for deployment, (4) Multi-agent support, (5) Battle-tested at Google, Ant Financial, and others. For research prototyping and small-scale experiments: SB3. It's simpler, better documented for individual use, and faster to iterate on.

---

### Q6: What is V-trace and why does IMPALA need it?
**A**: V-trace is an importance sampling correction for off-policy data. In IMPALA, actors collect data using their current (possibly stale) policy μ, but the learner trains on policy π which may have been updated since. Without correction, learning from stale data introduces bias. V-trace weights each transition by the importance sampling ratio `min(c̄, π(a|s)/μ(a|s))`, clipped to prevent extreme weights from causing instability. The clipping is the key innovation — it bounds the maximum correction while still correcting for policy lag.

## 10. Resources

### Official Docs
- **RLlib Docs**: https://docs.ray.io/en/latest/rllib/index.html
- **Ray GitHub**: https://github.com/ray-project/ray
- **Ray Tune**: https://docs.ray.io/en/latest/tune/index.html
- **RLlib Algorithms List**: https://docs.ray.io/en/latest/rllib/rllib-algorithms.html

### Video Tutorials
- **Ray Summit talks**: https://www.youtube.com/@RayDistributed
- **IMPALA paper explained**: https://www.youtube.com/watch?v=z2xRNbMqmQY
- **Multi-Agent RL (Berkeley CS285)**: https://www.youtube.com/playlist?list=PL_iWQOsE6TfVmKkQHucjPAoRtIJYt8a5A

### Research Papers
- **RLlib paper**: https://arxiv.org/abs/1712.09381
- **IMPALA**: https://arxiv.org/abs/1802.01561
- **MARL survey**: https://arxiv.org/abs/1911.10635
- **Population Based Training**: https://arxiv.org/abs/1711.09846

---

## Summary

| Concept | Takeaway |
|---------|----------|
| Why RLlib | Scales RL from laptop to cluster; multi-agent built-in |
| Architecture | Learner + RolloutWorkers + (ReplayBuffer) |
| Algorithm API | `AlgoConfig().environment().training().build()` |
| Ray Tune | Automated hyperparameter search; ASHA for efficiency |
| Multi-agent | Separate policies per agent type; policy mapping fn |
| vs SB3 | SB3 = simpler; RLlib = scalable & production-ready |

**You've completed the Reinforcement Learning module!**

Next: **08_Generative_AI_LLM** — Learn to build applications with GPT, LangChain, LlamaIndex, and vLLM!